# forex_rl_v4 — Colab GPU training (multi-contributor)

Runs a slice of the full walk-forward sweep (12 folds x 4 seeds = 48
fold/seed combos, covering all 16 years of data on disk) as concurrent
`train.py` processes on ONE Colab GPU, writing to a Google Drive folder
SHARED across everyone contributing — so multiple people's sessions combine
into one pool of results instead of each person's work being stuck on
their own Drive.

**Nothing to type, nothing to download or upload.** Open this from the
[Contribute tab](https://martin-gamcptrxwpxkksceimkyx3.streamlit.app/) link, accept the Google Drive access
prompt the next cell shows you, then `Runtime -> Change runtime type ->
GPU` and `Runtime -> Run all` — that's the whole setup. Your name is
generated for you (and remembered on your Drive for next time), the repo
is public so the code and historical price data pull straight from
GitHub, and your results push themselves straight back to GitHub in the
background as you train. No dashboard runs here, and none needed —
everyone's progress, including yours, shows up at the ONE shared dashboard:

**[https://martin-gamcptrxwpxkksceimkyx3.streamlit.app/](https://martin-gamcptrxwpxkksceimkyx3.streamlit.app/)**

**If the session disconnects:** re-run the notebook from the top (`Runtime
-> Run all`). You get the SAME generated name and the SAME shard indices
back, and each process's `--resume` skips fold/seed combos already
finished (by ANYONE, not just you) and resumes an interrupted fold from its
last mid-fold checkpoint. You'll lose whatever hadn't been auto-pushed yet
(up to ~5 minutes of progress), same as any other interruption.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Config — nothing to edit, just run this cell ===
# MY_NAME: auto-generated once and saved to YOUR OWN Drive (not the shared
# folder) so it's stable across reruns/reconnects — the claims registry and
# the dashboard's contributor leaderboard both key off this, so a name that
# changed every run would look like a new person abandoning old shards each
# time instead of one person resuming. Delete the file below if you ever
# want a fresh name.
import os
import random

_NAME_FILE = '/content/drive/MyDrive/.forex_rl_v4_my_name.txt'
if os.path.exists(_NAME_FILE):
    with open(_NAME_FILE) as f:
        MY_NAME = f.read().strip()
else:
    _ADJECTIVES = ['swift', 'sharp', 'bold', 'lucky', 'steady', 'silent', 'quick', 'wild',
                   'calm', 'fierce', 'golden', 'iron', 'midnight', 'rogue', 'stealth', 'turbo',
                   'rapid', 'clever', 'brave', 'sly']
    _NOUNS = ['falcon', 'bull', 'bear', 'pip', 'candle', 'trader', 'wolf', 'hawk', 'tiger',
              'shark', 'viper', 'eagle', 'ninja', 'comet', 'rocket', 'panther', 'cobra',
              'phoenix', 'lynx', 'scalper']
    MY_NAME = f'{random.choice(_ADJECTIVES)}-{random.choice(_NOUNS)}-{random.randint(10, 99)}'
    with open(_NAME_FILE, 'w') as f:
        f.write(MY_NAME)

# N_SHARDS_WANTED: how many shard slots to claim this session — one Colab
# GPU comfortably runs 4 in parallel (the project's standard allocation).
N_SHARDS_WANTED = 4

# TOTAL_SHARDS: total shard count across EVERYONE combined — the
# denominator for train.py's round-robin split. Must match what everyone
# else in the group is using. 48 = 12 folds x 4 seeds, i.e. one shard per
# fold/seed combo (the most parallelism this sweep can actually use).
TOTAL_SHARDS = 48

# SHARED_FOLDER: the group's shared Drive folder. Drive allows duplicate
# folder names, and a stale bookmark or a mis-clicked "Add shortcut to
# Drive" can silently point someone at a DIFFERENT folder that merely
# LOOKS the same in a file listing — which actually happened to a real
# contributor: isolated claims.json (so shard assignments collided with
# someone else's — both ended up training the exact same combos), no
# token files visible even though the owner had definitely added them.
#
# Writing through Drive's ID-anchored mount path
# (/content/drive/.shortcut-targets-by-id/<id>) turned out to fail with
# OSError [Errno 95] Operation not supported — that path is read-only in
# practice for this kind of file creation, so SHARED_FOLDER stays the
# normal named MyDrive path (proven to support writes) and the ID is used
# only as a READ-ONLY oracle below: a canary file written through the
# named path must become visible through the ID path within a few
# seconds, or this isn't really the shared folder.
import time

SHARED_FOLDER_ID = '1KKSoBLl8CurDqFtUz3UM9h_9fmDFDiqe'
SHARED_FOLDER = '/content/drive/MyDrive/forex_rl_v4_shared_results'
_id_path = f'/content/drive/.shortcut-targets-by-id/{SHARED_FOLDER_ID}'


def _wait_for_dir(path, seconds=30):
    # A freshly-added shortcut (or a just-established Drive mount) can take
    # a while to fully sync — give it real time before concluding it's
    # actually missing, rather than failing on the very first check.
    for _ in range(seconds):
        if os.path.isdir(path):
            return True
        time.sleep(1)
    return False


if not _wait_for_dir(SHARED_FOLDER):
    raise RuntimeError(
        f'{SHARED_FOLDER} not found after waiting 30s -- open the shared folder '
        f'link from the Contribute tab and click "Add shortcut to Drive" (if you '
        f'just did, Drive may still be syncing — wait a minute and re-run this cell).'
    )
if not _wait_for_dir(_id_path):
    raise RuntimeError(
        f'Cannot see the shared folder by its known Drive ID ({SHARED_FOLDER_ID}) '
        f'after waiting 30s -- your account may not actually have access yet. Open '
        f'the shared folder link from the Contribute tab, confirm you can see its '
        f'contents, then re-run this cell.'
    )

# Cross-view write check, downgraded from a hard failure to a warning:
# it caught the original wrong-folder incident correctly, but has since
# also fired for at least one contributor whose setup was verified
# correct by hand -- Colab's two Drive mount views (named path vs
# ID-anchored path) appear to cache independently, so a write on one side
# isn't guaranteed to ever become visible on the other within a single
# process's lifetime, real folder or not. A false block is worse than a
# missed warning here: get_shard_activity_table() on the dashboard's
# Training page shows exactly who is training which (fold, seed) combo,
# so a real duplicate is now visible and fixable after the fact instead
# of needing to be caught perfectly in advance.
_canary_name = f'.canary_{MY_NAME}.txt'
with open(os.path.join(SHARED_FOLDER, _canary_name), 'w') as f:
    f.write('ok')
_verified = False
for _ in range(15):
    if os.path.exists(os.path.join(_id_path, _canary_name)):
        _verified = True
        break
    time.sleep(1)
os.remove(os.path.join(SHARED_FOLDER, _canary_name))
if not _verified:
    print(
        f'WARNING: could not confirm {SHARED_FOLDER} is the real shared folder '
        f'(a write there is not showing up via its known Drive ID -- this can be '
        f'a genuine wrong-folder situation, OR just Drive mount caching; it is not '
        f'conclusive either way). Proceeding anyway. If your shard claims turn out '
        f'to collide with someone else\'s, check the dashboard\'s Training page '
        f'("Who\'s training what") -- if you see it happen, remove the '
        f'"forex_rl_v4_shared_results" shortcut and re-add it fresh from the '
        f'Contribute tab link.'
    )

# ITERATIONS: PPO iterations per fold/seed combo.
ITERATIONS = 100

print(f'You are: {MY_NAME}')
print(f'Wants {N_SHARDS_WANTED} of {TOTAL_SHARDS} total shards, '
      f'{ITERATIONS} iterations/combo, writing to {SHARED_FOLDER}')

In [ ]:
# Clone the code straight from GitHub — no zip, no manual upload. The repo
# is public, so this works with no authentication.
import os

PROJECT_DIR = '/content/martin'
REPO_URL = 'https://github.com/samdotbin/martin.git'

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
else:
    print(f'{PROJECT_DIR} already exists — pulling the latest instead of re-cloning.')
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}

In [ ]:
# data/raw's 182MB of historical CSVs isn't in git (see the repo's
# .gitignore) — it's attached to a GitHub Release instead. Downloads once;
# safe to re-run (skips if already present, e.g. after a disconnect where
# PROJECT_DIR survived but a fresh clone would still need this).
import urllib.request
import zipfile

DATA_URL = 'https://github.com/samdotbin/martin/releases/download/data-v1/forex_rl_v4_data_raw.zip'
data_dir = f'{PROJECT_DIR}/data/raw'

if os.path.isdir(data_dir) and len([f for f in os.listdir(data_dir) if f.endswith('.csv')]) >= 20:
    print(f'{data_dir} already has the data — skipping download.')
else:
    os.makedirs(data_dir, exist_ok=True)
    print('downloading historical price data (~43MB)...')
    tmp_zip = f'{data_dir}/_download.zip'
    urllib.request.urlretrieve(DATA_URL, tmp_zip)
    with zipfile.ZipFile(tmp_zip) as z:
        z.extractall(data_dir)
    os.remove(tmp_zip)
    n_csvs = len([f for f in os.listdir(data_dir) if f.endswith('.csv')])
    print(f'done — {n_csvs} CSV(s) in {data_dir}')

In [ ]:
# MetaTrader5 is Windows-only and gated by an environment marker in
# requirements.txt (`; platform_system == "Windows"`) — pip skips it
# automatically here, and nothing in the training path imports it.
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU visible. Runtime -> Change runtime type -> GPU, then re-run this cell."
)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Sanity check first — same coverage report you'd run locally.
!python data_pipeline.py

## Optional: bump ROLLOUT_N_ENVS for GPU

`config.ROLLOUT_N_ENVS` (default 32) sets how many environments are batched
into one model forward call. The model is small (2-5M params) and Colab GPUs
have plenty of headroom, so a higher value amortizes kernel-launch overhead
further. Edit `config.py` directly (or uncomment below) if you want to try
64 or 128 — there's no single right answer, and changing it does change the
exact (though not the statistical) outcome for a given seed, so pick a value
and keep it fixed for the whole sweep.

In [ ]:
# import config
# print('current ROLLOUT_N_ENVS:', config.ROLLOUT_N_ENVS)
# Edit config.py's ROLLOUT_N_ENVS value directly instead of monkey-patching
# here, since train.py runs as a subprocess below and won't see an in-notebook
# variable change.

In [ ]:
# Claim your shard indices from the shared registry — the system figures
# out which ones are still free, you don't need to be handed a range
# manually. Safe to re-run: idempotent for the SAME MY_NAME (returns your
# existing claims first, only grabs new ones for any shortfall).
import sys
sys.path.insert(0, PROJECT_DIR)
from scripts.claim_shards import claim

MY_SHARDS = claim(SHARED_FOLDER, TOTAL_SHARDS, N_SHARDS_WANTED, MY_NAME)
print(f'{MY_NAME}: claimed shards {MY_SHARDS}')

In [ ]:
import subprocess
import time
from datetime import datetime, timezone

# Each of YOUR processes gets its own subfolder under the SHARED Drive
# folder (via env var overrides — see config.py) so concurrent processes —
# yours or anyone else's, in case sessions overlap — never read-modify-write
# the SAME RUN_MANIFEST.json at once. All processes share the one local
# code+data checkout (PROJECT_DIR) — data/raw is read-only.
# (No os.makedirs(SHARED_FOLDER, ...) here on purpose — the config cell
# above already verified it exists. Creating it here-if-missing is exactly
# what silently papers over pointing at the wrong folder.)

SESSION_START_TIME = datetime.now(timezone.utc).isoformat()
procs = []

def _launch_shard(idx):
    shard_dir = f'{SHARED_FOLDER}/shard{idx}'
    ckpt_dir = f'{shard_dir}/checkpoints'
    runs_dir = f'{shard_dir}/runs'
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(runs_dir, exist_ok=True)

    env = os.environ.copy()
    env['FOREX_RL_CHECKPOINT_DIR'] = ckpt_dir
    env['FOREX_RL_RUNS_DIR'] = runs_dir

    log_path = f'{shard_dir}/train_log.txt'
    p = subprocess.Popen(
        ['python', 'train.py', '--resume', '--iterations', str(ITERATIONS),
         '--shard-index', str(idx), '--shard-count', str(TOTAL_SHARDS)],
        cwd=PROJECT_DIR, env=env,
        stdout=open(log_path, 'w'), stderr=subprocess.STDOUT,
        # Without this, a shard inherits the notebook kernel's process
        # group — Colab's "Interrupt execution" (e.g. stopping the
        # auto-refreshing monitor cell below) sends SIGINT to the WHOLE
        # group, killing every shard along with it. start_new_session=True
        # makes each shard its own session leader so only the kernel gets
        # interrupted, never the training.
        start_new_session=True,
    )
    entry = {'idx': idx, 'proc': p, 'log': log_path}
    procs.append(entry)
    print(f'started shard {idx} (pid {p.pid}) -> {log_path}')
    return entry

for idx in MY_SHARDS:
    _launch_shard(idx)
    time.sleep(2)  # stagger starts slightly so they don't all hit disk/data-loading at once

print(f'\n{len(procs)} shard(s) running in the background. Use the next cell '
      f'(re-run it any time) to check progress. Finished shards get replaced '
      f'automatically (see the background loops below) as long as unclaimed '
      f'shards remain.')

In [ ]:
# Four independent background loops, built on scripts/periodic.py's
# run_forever() — it owns the "never let one bad cycle kill the thread"
# shape once (a missing token, a network blip, a transient API error just
# prints and retries next cycle), so each loop below is just "what to do,
# how often" with no boilerplate to get wrong or forget:
#  - checkpoint files to GitHub every 5 min (slow on purpose — checkpoints
#    only change every ~10% of iterations, so pushing more often would
#    mostly add redundant commits for unchanged files)
#  - a tiny heartbeat every 60s — what makes the owner's dashboard show you
#    as online with no publish step on their part
#  - an optional Telegram ping every 60s, in its OWN thread so a Telegram
#    hiccup can never delay or block the heartbeat (or vice versa)
#  - a shard-refill check every 30s — when one of your shards finishes
#    cleanly, this claims and launches a replacement automatically, so your
#    GPU doesn't sit idle for the rest of the session just because its
#    first assignment finished before the whole sweep did
# Tokens are read fresh EVERY cycle (not once at startup) — if the owner
# adds a token file after training has already started, the very next
# cycle picks it up with nothing to re-run.
import re
import sys
import threading
import time

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
from scripts.periodic import run_forever
from scripts.push_to_github import push_all, push_heartbeat, read_shared_token
from scripts.notify_telegram import read_telegram_config, send_message
from scripts.claim_shards import claim

GITHUB_REPO = 'samdotbin/martin'
PUSH_INTERVAL_SECONDS = 300   # 5 minutes — checkpoint files
FAST_INTERVAL_SECONDS = 60    # 1 minute — heartbeat + Telegram
REFILL_INTERVAL_SECONDS = 30  # how often to check for a finished shard to replace
_CKPT_NAME_RE = re.compile(r'^fold_(\d+)_seed_(-?\d+)_(?:latest|best)(?:_regime)?\.')

def _gather_shard_status():
    """One entry per shard this session owns — running/exited, which
    (fold, seed) it's on (read from whichever checkpoint file has shown up
    for it, since that's the same ground truth train.py itself writes —
    no separate combo-math to keep in sync), and its log's last line. Feeds
    BOTH the heartbeat (structured, for the dashboard) and the Telegram
    ping (formatted as text) so there's exactly one place this logic lives."""
    out = []
    for entry in procs:
        rc = entry['proc'].poll()
        status = 'running' if rc is None else f'exited (code {rc})'
        last_line = None
        if os.path.exists(entry['log']):
            with open(entry['log']) as f:
                non_empty = [l.strip() for l in f if l.strip()]
            if non_empty:
                last_line = non_empty[-1]
        fold_id = seed = None
        ckpt_dir = f'{SHARED_FOLDER}/shard{entry["idx"]}/checkpoints'
        if os.path.isdir(ckpt_dir):
            for fname in os.listdir(ckpt_dir):
                m = _CKPT_NAME_RE.match(fname)
                if m:
                    fold_id, seed = int(m.group(1)), int(m.group(2))
                    break
        out.append({
            'shard': entry['idx'], 'status': status,
            'fold_id': fold_id, 'seed': seed, 'last_log': last_line,
        })
    return out

def _push_checkpoints():
    token = read_shared_token(SHARED_FOLDER)
    pushed = push_all(GITHUB_REPO, token, MY_NAME, SHARED_FOLDER, MY_SHARDS)
    if pushed:
        print(f'[{time.strftime("%H:%M:%S")}] pushed {len(pushed)} file(s) to GitHub')

def _push_heartbeat():
    token = read_shared_token(SHARED_FOLDER)
    push_heartbeat(GITHUB_REPO, token, MY_NAME, MY_SHARDS,
                   shard_status=_gather_shard_status(), session_started_at=SESSION_START_TIME)

_telegram_warned = False

def _push_telegram():
    global _telegram_warned
    tg_token, chat_id = read_telegram_config(SHARED_FOLDER)
    if not (tg_token and chat_id):
        if not _telegram_warned:
            print('(telegram not configured yet — skipping pings; see scripts/notify_telegram.py)')
            _telegram_warned = True
        return
    lines = [f'{MY_NAME} — {len(procs)} shard(s):']
    for s in _gather_shard_status():
        combo = f"fold {s['fold_id']}/seed {s['seed']}" if s['fold_id'] is not None else 'starting up'
        lines.append(f"shard {s['shard']} ({combo}) — {s['status']}: {s['last_log'] or ''}")
    send_message(tg_token, chat_id, '\n'.join(lines))

_total_wanted = [len(MY_SHARDS)]  # mutable box: how many shards we've asked for, ever

def _refill_finished_shards():
    """A shard that finishes (exits code 0) used to just sit there idle for
    the rest of the session — nothing replaced it, so a contributor's GPU
    capacity went unused for whatever remained of the sweep unless they
    noticed and manually re-ran the claim+launch cells. This checks for
    newly-finished shards and claims + launches exactly one replacement per
    one that finished, keeping the concurrently-running count steady at
    what was originally requested (not growing without bound). Only
    replaces CLEAN exits (code 0) — a crash (nonzero exit) is left alone
    rather than auto-retried, so a real bug doesn't get silently masked by
    endless auto-relaunching."""
    for entry in list(procs):
        if entry['proc'].poll() == 0 and not entry.get('_refilled'):
            entry['_refilled'] = True
            _total_wanted[0] += 1
            try:
                current = claim(SHARED_FOLDER, TOTAL_SHARDS, _total_wanted[0], MY_NAME)
            except RuntimeError as e:
                print(f'shard {entry["idx"]} finished — no more shards available to replace it with: {e}')
                continue
            have = {e['idx'] for e in procs}
            new_idx = [i for i in current if i not in have]
            for idx in new_idx:
                print(f'shard {entry["idx"]} finished cleanly — claimed and starting replacement shard {idx}')
                _launch_shard(idx)
                time.sleep(2)

def _start_loop(thread_var_name, target, args, label):
    existing = globals().get(thread_var_name)
    if existing is not None and existing.is_alive():
        print(f'{label} already running — not starting a second thread.')
        return existing
    t = threading.Thread(target=target, args=args, daemon=True)
    t.start()
    print(f'{label} started.')
    return t

_auto_push_thread = _start_loop(
    '_auto_push_thread', run_forever, (_push_checkpoints, PUSH_INTERVAL_SECONDS, 'checkpoint push'),
    f'background checkpoint push (every {PUSH_INTERVAL_SECONDS // 60} min)',
)
_heartbeat_thread = _start_loop(
    '_heartbeat_thread', run_forever, (_push_heartbeat, FAST_INTERVAL_SECONDS, 'heartbeat'),
    f'heartbeat loop (every {FAST_INTERVAL_SECONDS}s)',
)
_telegram_thread = _start_loop(
    '_telegram_thread', run_forever, (_push_telegram, FAST_INTERVAL_SECONDS, 'telegram'),
    f'telegram loop (every {FAST_INTERVAL_SECONDS}s once configured)',
)
_refill_thread = _start_loop(
    '_refill_thread', run_forever, (_refill_finished_shards, REFILL_INTERVAL_SECONDS, 'shard refill'),
    f'shard refill loop (every {REFILL_INTERVAL_SECONDS}s — replaces finished shards automatically)',
)
print('Leave this tab open — closing it stops these threads (your last push still counts).')

In [ ]:
# Auto-refreshing progress monitor — updates itself, no need to re-run.
# All local, no network calls (just reading your own shards' log files and
# nvidia-smi), so this can refresh often without costing anything. Stop it
# any time (Runtime -> Interrupt execution, or the cell's stop button) to
# free up the notebook for other cells — your training keeps running in
# the background either way, since it's a separate subprocess, not this
# cell. Re-run this cell to resume watching.
import time
from IPython.display import clear_output

MONITOR_REFRESH_SECONDS = 20

try:
    while True:
        clear_output(wait=True)
        all_done = True
        for entry in procs:
            rc = entry['proc'].poll()
            status = 'running' if rc is None else f'exited (code {rc})'
            if rc is None:
                all_done = False
            print(f"shard {entry['idx']} (pid {entry['proc'].pid}): {status}")
            !tail -n 3 "{entry['log']}"
            print()

        # Resource check — see the parallelism config cell for what to watch for.
        !nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv
        !uptime
        print(f"\n(refreshing every {MONITOR_REFRESH_SECONDS}s — interrupt this cell any time)")

        if all_done:
            print("\nAll shards finished! Move on to the merge cell at the bottom.")
            break
        time.sleep(MONITOR_REFRESH_SECONDS)
except KeyboardInterrupt:
    print("\nStopped watching (training keeps running in the background) — re-run this cell to resume.")

## Merge and archive

Every contributor's process wrote to their own `shard{i}/` subfolder under
the ONE shared Drive folder. This pulls in ALL `TOTAL_SHARDS` of them —
everyone's combined progress, not just yours — into one consolidated
`checkpoints/`+`runs/` in `PROJECT_DIR`, and zips that up for download.
Safe to run any time, by anyone with access to the shared folder, even
mid-training.

In [ ]:
import datetime

all_shard_dirs = ' '.join(f'{SHARED_FOLDER}/shard{idx}' for idx in range(TOTAL_SHARDS))
!python scripts/merge_shard_results.py {all_shard_dirs}

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
archive_path = f'/content/drive/MyDrive/forex_rl_v4_merged_results_{stamp}.zip'
!zip -rq "{archive_path}" checkpoints runs
print('wrote', archive_path)